# \# ANN (Artificial Neural Network)

## \# ANN for Regression

In [ ]:
import pandas as pd
import numpy as np

### ->  Load Data

In [ ]:
df = pd.read_csv("powerplant_data.csv")
df.head()

In [ ]:
# AT -> temperature
# V -> vaccum
# AP -> Pressure
# RH -> Humidity
# PE -> produced energy

In [ ]:
df.isnull().sum()

In [ ]:
X = df.drop("PE", axis = 1)
y = df["PE"]

In [ ]:
# Split data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

In [ ]:
# scale
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# converting data into tensors
import torch
import torch.nn as nn

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [ ]:
# TensorDataset and DataLoader

from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# Deep Learning

In [ ]:
# define ANN Model

class ANN(nn.Module):
    
    def __init__(self):
        super(ANN, self).__init__()
        
        self.model = nn.Sequential(
            # 1st hidden layer
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),

            #2nd hidden layer
            nn.Linear(6,6),
            nn.ReLU(),

            #output layer
            nn.Linear(6,1),
        )


    def forward(self, x):
        return self.model(x)

In [ ]:
# building ANN model

import torch.optim as optim

model = ANN()

# loss -> MSE, optimizer
crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
# train the ANN
train_losses = []
val_losses = []
best_val_loss = float("inf")

epochs = 50
for epoch in range(epochs):
    model.train()
    running_loss = 0.0 # total training loss for 1 epoch
    # xb -> features of 1 batch
    # yb -> labels of 1 batch
    for xb, yb in train_loader:
        optimizer.zero_grad() # to refresh the gradient
        outputs = model(xb) # forward propagation-> predicted outputs for this batch
        loss = crietrion(outputs, yb) # compute loss
        loss.backward() # backward propagation -> compute gradient
        optimizer.step() # params update

        running_loss += loss.item() # loss is a tensor coverting into py float

    epoch_train_loss = running_loss/len(train_loader)
    train_losses.append(epoch_train_loss)

    # validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad() : # no gradients compute
        for xb, yb in test_loader:
            outputs = model(xb)
            loss = crietrion(outputs, yb)
            running_val_loss += loss.item() 

    epoch_val_loss = running_val_loss/ len(test_loader)
    val_losses.append(epoch_val_loss)


    print(f"epoch {epoch+1}/{epochs} ===> train loss = {epoch_train_loss} & val loss = {epoch_val_loss}")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pt") # .pt or .pth
    

In [ ]:
import matplotlib.pyplot as plt

loss_df = pd.DataFrame({
    "training_loss" : train_losses,
    "validation_loss" : val_losses
})

plt.figure(figsize=(8,6))
plt.plot(loss_df["training_loss"], label = "training_loss")
plt.plot(loss_df["validation_loss"], label = "validation_loss")
plt.xlabel("Epochs")
plt.ylabel("Losses")
plt.title("visualization of losses per epoch")
plt.legend()
plt.show()

In [ ]:
# loading the best model
model.load_state_dict(torch.load("best_model.pt"))

In [ ]:
# evaluate our model

model.eval()
with torch.no_grad():
    train_pred = model(X_train_tensor)
    test_pred  = model(X_test_tensor)

    train_mse_loss = crietrion(train_pred, y_train_tensor)
    test_mse_loss = crietrion(test_pred, y_test_tensor)

print("Training MSE : ", train_mse_loss.item())
print("Testing MSE : ", test_mse_loss.item())


In [ ]:
from sklearn.metrics import r2_score

print("r2 socre : ", r2_score(y_test, test_pred))

In [ ]:
pred_df = pd.DataFrame(test_pred.numpy(), columns=["Predicted values"])
actual_df = pd.DataFrame(y_test.values, columns = ["actual values"])

pd.concat([pred_df, actual_df], axis=1)